# DEPURACION Y AGRUPACIÓN DE DEFUNCIONES

## 1. IMPORTAR LIBRERIAS

In [1]:
### IMPORTAR LIBRERIAS ###

import os
import warnings
import pandas as pd
import pyarrow

warnings.filterwarnings("ignore")
print(f"pandas  : {pd.__version__}")
print(f"pyarrow : {pyarrow.__version__}")

pandas  : 2.3.3
pyarrow : 23.0.1


## 2. CONFIGURACIÓN DE RUTAS Y CONSTANTES

In [2]:
# ── Rutas ──────────────────────────────────────────────────────────────────────
RUTA_CAMPOS = (
    "/kaggle/input/datasets/nicolasacostaa/campos-defunciones/Campos_Defunciones.xlsx"
)
RUTA_DEFUN  = "/kaggle/input/datasets/nicolasacostaa/defunciones/"
RUTA_SALIDA = "/kaggle/working/defunciones_consolidadas.parquet"

# ── Rango de años a procesar ───────────────────────────────────────────────────
ANIO_INICIO = 1979
ANIO_FIN    = 2024

# ── Nombre de la columna de conteo ────────────────────────────────────────────
COLUMNA_FALLECIDOS = "Número de Fallecidos"

print("Configuración cargada ✓")
print(f"  Rango        : {ANIO_INICIO} – {ANIO_FIN}")
print(f"  Homologación : {RUTA_CAMPOS}")
print(f"  Defunciones  : {RUTA_DEFUN}")
print(f"  Salida       : {RUTA_SALIDA}")


Configuración cargada ✓
  Rango        : 1979 – 2024
  Homologación : /kaggle/input/datasets/nicolasacostaa/campos-defunciones/Campos_Defunciones.xlsx
  Defunciones  : /kaggle/input/datasets/nicolasacostaa/defunciones/
  Salida       : /kaggle/working/defunciones_consolidadas.parquet


## 3. FUNCIONES DE HOMOLOGACIÓN

### 3.1. cargar_homologacion(ruta_campos)

In [3]:
def cargar_homologacion(ruta_campos: str) -> list[dict]:
    """
    Lee el archivo Excel de homologación y devuelve una lista de configuraciones,
    una por cada rango de años definido en el archivo.

    Parámetros
    ----------
    ruta_campos : str
        Ruta al archivo Campos_Defunciones.xlsx.

    Retorna
    -------
    list[dict]
        Lista de dicts con las llaves:
          - 'anio_inicio' (int)
          - 'anio_fin'    (int)
          - 'tipo'        (str)  → 'csv' o 'txt'
          - 'mapeo'       (dict) → {nombre_local: nombre_global}
            Solo incluye campos con nombre global definido (no NaN).

    Ejemplo
    -------
    >>> configs = cargar_homologacion(RUTA_CAMPOS)
    >>> configs[0]
    {'anio_inicio': 1979, 'anio_fin': 1991, 'tipo': 'txt',
     'mapeo': {'COD_DPTO': 'CÓDIGO DEPARTAMENTO', ...}}
    """
    df_raw = pd.read_excel(ruta_campos, sheet_name=0, header=None)

    # Fila 0 → nombres globales (columna E en adelante = índice 4+)
    nombres_globales = df_raw.iloc[0, 4:].tolist()

    configuraciones = []
    for _, fila in df_raw.iloc[1:].iterrows():
        anio_inicio  = int(fila.iloc[1])
        anio_fin     = int(fila.iloc[2])
        tipo_archivo = str(fila.iloc[3]).strip().lower()
        nombres_locales = fila.iloc[4:].tolist()

        mapeo = {}
        for nombre_global, nombre_local in zip(nombres_globales, nombres_locales):
            if pd.notna(nombre_global) and pd.notna(nombre_local):
                ng = str(nombre_global).strip()
                nl = str(nombre_local).strip()
                if ng and nl:
                    mapeo[nl] = ng

        configuraciones.append({
            "anio_inicio": anio_inicio,
            "anio_fin":    anio_fin,
            "tipo":        tipo_archivo,
            "mapeo":       mapeo,
        })

    return configuraciones


print("Función 'cargar_homologacion' definida ✓")

Función 'cargar_homologacion' definida ✓


### 3.2 `obtener_config_para_anio(anio, configuraciones)`

In [4]:
def obtener_config_para_anio(anio: int, configuraciones: list[dict]) -> dict | None:
    """
    Busca la configuración de homologación aplicable a un año específico.

    Parámetros
    ----------
    anio : int
        Año a buscar (ej. 2005).
    configuraciones : list[dict]
        Lista devuelta por cargar_homologacion().

    Retorna
    -------
    dict | None
        La configuración del rango que contiene ese año,
        o None si no existe ningún rango aplicable.

    Ejemplo
    -------
    >>> config = obtener_config_para_anio(2005, configuraciones)
    >>> config['tipo']
    'txt'
    """
    for config in configuraciones:
        if config["anio_inicio"] <= anio <= config["anio_fin"]:
            return config
    return None


print("Función 'obtener_config_para_anio' definida ✓")

Función 'obtener_config_para_anio' definida ✓


## 4. FUNCIONES DE LECTURA Y HOMOLOGACIÓN DE ARCHIVOS

### 4.1 `leer_archivo_defunciones(ruta_archivo, tipo)`

In [5]:
def buscar_archivo_defunciones(ruta_defun: str, anio: int, tipo: str) -> str | None:
    """
    Busca el archivo de defunciones de un año dado probando distintas
    variantes de nombre y extensión.

    Parámetros
    ----------
    ruta_defun : str   Carpeta donde están los archivos.
    anio       : int   Año buscado.
    tipo       : str   Extensión preferida según el Excel ('csv' o 'txt').

    Retorna
    -------
    str | None   Ruta completa al archivo encontrado, o None si no existe.

    Variantes probadas (en orden)
    ------------------------------
    Defun{anio}.{tipo}   Defun{anio}.{tipo upper}
    defun{anio}.{tipo}   defun{anio}.{tipo upper}
    Defun{anio}.csv      Defun{anio}.txt
    DEFUN{anio}.{tipo}   DEFUN{anio}.CSV  DEFUN{anio}.TXT
    """
    candidatos = []
    for prefijo in [f"Defun{anio}", f"defun{anio}", f"DEFUN{anio}"]:
        for ext in [tipo, tipo.upper(), "csv", "txt", "CSV", "TXT"]:
            candidatos.append(os.path.join(ruta_defun, f"{prefijo}.{ext}"))
    # Eliminar duplicados preservando orden
    vistos = set()
    candidatos_unicos = []
    for c in candidatos:
        if c not in vistos:
            vistos.add(c)
            candidatos_unicos.append(c)

    for ruta in candidatos_unicos:
        if os.path.exists(ruta):
            return ruta
    return None


def detectar_separador(ruta_archivo: str, encoding: str) -> str:
    """
    Detecta el separador de columnas leyendo las primeras 3 líneas del archivo
    y eligiendo el carácter más consistente entre ellas.

    Parámetros
    ----------
    ruta_archivo : str   Ruta al archivo.
    encoding     : str   Encoding a usar para la lectura.

    Retorna
    -------
    str   Separador detectado (';', ',', '|' o '\t').
          Devuelve ';' como fallback si no se detecta ninguno.

    Lógica
    ------
    Para cada separador candidato se comprueba que:
      a) Aparezca en la línea de cabecera.
      b) El número de ocurrencias sea consistente entre las primeras 3 líneas
         (varianza ≤ 1), lo que indica que es el separador real y no ruido.
    Se elige el separador con mayor número de columnas que cumpla lo anterior.
    """
    separadores_candidatos = [";", ",", "|", "\t"]
    try:
        with open(ruta_archivo, "r", encoding=encoding, errors="replace") as f:
            lineas = [f.readline() for _ in range(3)]
        lineas = [l for l in lineas if l.strip()]
    except Exception:
        return ";"

    mejor_sep   = ";"
    mejor_ncols = 0

    for sep in separadores_candidatos:
        conteos = [l.count(sep) for l in lineas]
        if conteos[0] == 0:
            continue
        # Consistencia: diferencia máxima entre líneas ≤ 1
        if max(conteos) - min(conteos) <= 1:
            ncols = conteos[0] + 1
            if ncols > mejor_ncols:
                mejor_ncols = ncols
                mejor_sep   = sep

    return mejor_sep


def leer_archivo_defunciones(ruta_archivo: str, tipo: str) -> pd.DataFrame:
    """
    Lee un archivo de defunciones detectando automáticamente encoding y separador.

    Parámetros
    ----------
    ruta_archivo : str   Ruta completa al archivo.
    tipo         : str   'csv' o 'txt' (informativo).

    Retorna
    -------
    pd.DataFrame   Todas las columnas como string.

    Normalización de cabecera
    --------------------------
    Los nombres de columna se normalizan siempre a MAYÚSCULAS después de
    limpiar espacios y el carácter BOM (\ufeff). Esto garantiza que la
    comparación con el mapeo —que proviene del Excel en mayúsculas— funcione
    independientemente de cómo el archivo almacene su cabecera
    (minúsculas, mixto, etc.).

    Estrategia de lectura
    ----------------------
    1. Detectar encoding con chardet (muestra 100 KB).
    2. Detectar separador con detectar_separador().
    3. Intentar lectura con header=0 (primera fila es cabecera).
    4. Si el DataFrame queda con 1 sola columna (separador no aplicó),
       reintentar con los demás separadores candidatos.
    5. Normalizar nombres de columna: strip + lstrip BOM + upper().
    """
    import chardet

    with open(ruta_archivo, "rb") as f:
        enc_info = chardet.detect(f.read(100_000))
    encoding = enc_info.get("encoding") or "latin-1"

    encodings_fallback = list(dict.fromkeys([encoding, "latin-1", "utf-8", "cp1252"]))
    separadores_orden  = [";", ",", "|", "\t"]

    for enc in encodings_fallback:
        sep = detectar_separador(ruta_archivo, enc)
        seps_a_probar = [sep] + [s for s in separadores_orden if s != sep]

        for s in seps_a_probar:
            try:
                df = pd.read_csv(
                    ruta_archivo,
                    sep=s,
                    dtype=str,
                    encoding=enc,
                    low_memory=False,
                    on_bad_lines="skip",
                )
                if len(df.columns) <= 1:
                    continue

                # ── Normalizar cabecera: strip + quitar BOM + MAYÚSCULAS ──────
                df.columns = [
                    str(c).strip().lstrip("\ufeff").upper()
                    for c in df.columns
                ]
                return df
            except Exception:
                continue

    raise ValueError(
        f"No se pudo leer el archivo con ningún encoding/separador: {ruta_archivo}"
    )


print("Funciones 'buscar_archivo_defunciones', 'detectar_separador' y")
print("'leer_archivo_defunciones' definidas ✓")
print()
print("🔑 Corrección aplicada: cabecera normalizada a MAYÚSCULAS")
print("   (garantiza match con el mapeo del Excel para todos los años)")


Funciones 'buscar_archivo_defunciones', 'detectar_separador' y
'leer_archivo_defunciones' definidas ✓

🔑 Corrección aplicada: cabecera normalizada a MAYÚSCULAS
   (garantiza match con el mapeo del Excel para todos los años)


### 4.2 `homologar_columnas(df, mapeo)`

In [6]:
def homologar_columnas(df: pd.DataFrame, mapeo: dict) -> pd.DataFrame:
    """
    Renombra las columnas de un DataFrame usando el mapeo de homologación,
    descartando columnas que no tienen correspondencia global.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame con nombres de columnas originales (del archivo fuente).
    mapeo : dict
        Diccionario {nombre_local → nombre_global}.

    Retorna
    -------
    pd.DataFrame
        DataFrame solo con las columnas presentes en el mapeo,
        renombradas a sus nombres globales.

    Ejemplo
    -------
    >>> df_hom = homologar_columnas(df_crudo, config['mapeo'])
    >>> df_hom.columns.tolist()
    ['CÓDIGO DEPARTAMENTO', 'CÓDIGO MUNICIPIO', ...]
    """
    cols_a_usar = {col: mapeo[col] for col in df.columns if col in mapeo}
    return df[list(cols_a_usar.keys())].rename(columns=cols_a_usar)


print("Función 'homologar_columnas' definida ✓")

Función 'homologar_columnas' definida ✓


## 5. PROCESAMIENTO POR AÑO

### 5.1. `procesar_anio(anio, configuraciones, ruta_defun, verbose)`

In [7]:
def procesar_anio(
    anio: int,
    configuraciones: list[dict],
    ruta_defun: str,
    ruta_tmp: str,
    verbose: bool = True,
) -> str | None:
    """
    Carga, homologa, agrupa y escribe a disco las defunciones de un año.

    En lugar de retornar un DataFrame (que acumula RAM), escribe directamente
    un archivo Parquet por año en la carpeta temporal ruta_tmp y retorna
    la ruta del archivo creado.

    Parámetros
    ----------
    anio             : int         Año a procesar (ej. 1985).
    configuraciones  : list[dict]  Resultado de cargar_homologacion().
    ruta_defun       : str         Carpeta con los archivos de defunciones.
    ruta_tmp         : str         Carpeta temporal donde se escribe el Parquet
                                   del año (ej. /kaggle/working/tmp_anios/).
    verbose          : bool        Muestra progreso. Default: True.

    Retorna
    -------
    str | None
        Ruta al Parquet del año escrito, o None si no se procesó.

    Estrategia de bajo consumo de RAM
    -----------------------------------
    - Se procesa un año a la vez: el DataFrame se crea, agrupa y vuelca a disco.
    - Después del to_parquet(), el DataFrame se elimina explícitamente con del
      y se llama gc.collect() para liberar la memoria de inmediato.
    - El llamador (consolidar_defunciones) nunca acumula DataFrames en memoria;
      solo guarda la lista de rutas de los Parquet parciales.
    """
    import gc

    config = obtener_config_para_anio(anio, configuraciones)
    if config is None:
        if verbose:
            print(f"  [OMITIDO] {anio}: sin configuración de homologación.")
        return None

    ruta_completa = buscar_archivo_defunciones(ruta_defun, anio, config["tipo"])
    if ruta_completa is None:
        if verbose:
            print(f"  [OMITIDO] {anio}: archivo no encontrado en {ruta_defun!r}.")
        return None

    try:
        df = leer_archivo_defunciones(ruta_completa, config["tipo"])

        if df.empty:
            if verbose:
                print(f"  [VACÍO]   {anio}: archivo leído pero sin filas.")
            return None

        cols_antes = set(df.columns)
        df = homologar_columnas(df, config["mapeo"])

        if df.empty or len(df.columns) == 0:
            if verbose:
                cols_mapeo = set(config["mapeo"].keys())
                coincid    = cols_antes & cols_mapeo
                print(f"  [VACÍO]   {anio}: 0 columnas tras homologar.")
                print(f"            Cols en archivo : {sorted(cols_antes)[:10]} ...")
                print(f"            Cols en mapeo   : {sorted(cols_mapeo)[:10]} ...")
                print(f"            Coincidencias   : {sorted(coincid)}")
            return None

        # Agrupar y contar (cada fila original = 1 fallecido)
        cols_grupo = list(df.columns)
        df = (
            df.fillna("")
              .groupby(cols_grupo, as_index=False, dropna=False)
              .size()
              .rename(columns={"size": COLUMNA_FALLECIDOS})
        )
        df[COLUMNA_FALLECIDOS] = df[COLUMNA_FALLECIDOS].astype("int32")

        total = int(df[COLUMNA_FALLECIDOS].sum())

        # ── Escribir a disco y liberar RAM de inmediato ────────────────────────
        os.makedirs(ruta_tmp, exist_ok=True)
        ruta_parquet_anio = os.path.join(ruta_tmp, f"defun_{anio}.parquet")
        df.to_parquet(ruta_parquet_anio, index=False, engine="pyarrow")

        del df
        gc.collect()

        if verbose:
            print(f"  [OK]      {anio}: {total:>10,} fallecidos | {len(pd.read_parquet(ruta_parquet_anio)):>10,} filas únicas")

        return ruta_parquet_anio

    except Exception as e:
        if verbose:
            print(f"  [ERROR]   {anio}: {e}")
        return None


print("Función 'procesar_anio' definida ✓")
print("  → Escribe cada año a Parquet en disco y libera RAM inmediatamente.")


Función 'procesar_anio' definida ✓
  → Escribe cada año a Parquet en disco y libera RAM inmediatamente.


## 6. CONSOLIDACIÓN Y EXPORTACIÓN A PARQUET

### 6.1. `consolidar_defunciones(...)`

In [8]:
def consolidar_defunciones(
    ruta_campos: str  = RUTA_CAMPOS,
    ruta_defun:  str  = RUTA_DEFUN,
    ruta_salida: str  = RUTA_SALIDA,
    anio_inicio: int  = ANIO_INICIO,
    anio_fin:    int  = ANIO_FIN,
    verbose:     bool = True,
) -> None:
    """
    Función principal. Procesa todos los años y consolida en un Parquet final
    usando escritura incremental para minimizar el uso de RAM.

    Parámetros
    ----------
    ruta_campos : str   Ruta al Excel de homologación.
    ruta_defun  : str   Carpeta con los archivos de defunciones.
    ruta_salida : str   Ruta del Parquet final de salida.
    anio_inicio : int   Primer año a procesar. Default: ANIO_INICIO.
    anio_fin    : int   Último año a procesar.  Default: ANIO_FIN.
    verbose     : bool  Muestra progreso. Default: True.

    Estrategia de bajo consumo de RAM
    -----------------------------------
    En lugar del flujo clásico (concat de todos los DataFrames → to_parquet),
    se usa un enfoque de tres pasos:

    1. POR AÑO: procesar_anio() agrupa y vuelca cada año a un Parquet parcial
       en /kaggle/working/tmp_defun_anios/ y libera la RAM con gc.collect().
       En memoria solo existe UN año a la vez.

    2. COMBINACIÓN: se usa pyarrow.dataset para leer todos los Parquets parciales
       como un dataset virtual (sin cargar nada en RAM) y escribir el Parquet
       final en streaming con un ParquetWriter. Cada fragmento se procesa y
       descarta uno a la vez.

    3. LIMPIEZA: se eliminan los Parquets parciales temporales.

    Retorna
    -------
    None   (el resultado queda en ruta_salida en disco).

    Para leer el resultado:
        df = pd.read_parquet("/kaggle/working/defunciones_consolidadas.parquet")
    """
    import gc
    import shutil
    import pyarrow         as pa
    import pyarrow.dataset as ds
    import pyarrow.parquet as pq

    RUTA_TMP = os.path.join(os.path.dirname(ruta_salida), "tmp_defun_anios")

    if verbose:
        print("=" * 62)
        print("  PROCESAMIENTO DE DEFUNCIONES — COLOMBIA")
        print("=" * 62)
        print(f"  Rango        : {anio_inicio} – {anio_fin}")
        print(f"  Homologación : {ruta_campos}")
        print(f"  Defunciones  : {ruta_defun}")
        print(f"  Salida       : {ruta_salida}")
        print(f"  Tmp parciales: {RUTA_TMP}")
        print("=" * 62)

    # ── PASO 1: cargar homologación ────────────────────────────────────────────
    if verbose:
        print("\n[1/3] Cargando archivo de homologación...")
    configuraciones = cargar_homologacion(ruta_campos)
    if verbose:
        print(f"      {len(configuraciones)} rangos de formato encontrados.")

    # ── PASO 2: procesar año por año → Parquet parcial en disco ───────────────
    if verbose:
        print("\n[2/3] Procesando archivos por año (un año en RAM a la vez)...")
        print(f"  {'AÑO':<10} {'ESTADO':<12} {'FALLECIDOS':>15} {'FILAS ÚNICAS':>14}")
        print(f"  {'-'*8:<10} {'-'*10:<12} {'-'*13:>15} {'-'*12:>14}")

    rutas_parciales = []
    for anio in range(anio_inicio, anio_fin + 1):
        ruta_parquet = procesar_anio(
            anio, configuraciones, ruta_defun, RUTA_TMP, verbose
        )
        if ruta_parquet:
            rutas_parciales.append(ruta_parquet)
        gc.collect()

    if not rutas_parciales:
        raise ValueError(
            "No se encontraron datos. Verifica las rutas y los archivos."
        )

    # ── PASO 3: combinar Parquets con PyArrow en streaming (sin cargar a RAM) ──
    if verbose:
        print(f"\n[3/3] Combinando {len(rutas_parciales)} archivos parciales → Parquet final...")
        print("      (lectura en streaming, sin acumular en RAM)")

    os.makedirs(os.path.dirname(ruta_salida) or ".", exist_ok=True)

    dataset   = ds.dataset(rutas_parciales, format="parquet")
    schema    = dataset.schema
    writer    = pq.ParquetWriter(ruta_salida, schema, compression="snappy")
    total_filas = 0
    total_fall  = 0

    for batch in dataset.to_batches(batch_size=500_000):
        writer.write_batch(batch)
        total_filas += batch.num_rows
        col_idx = schema.get_field_index(COLUMNA_FALLECIDOS)
        if col_idx >= 0:
            total_fall += batch.column(col_idx).to_pylist().__len__()

    writer.close()

    # Contar fallecidos reales leyendo solo esa columna
    total_fall = pq.read_table(
        ruta_salida, columns=[COLUMNA_FALLECIDOS]
    ).column(COLUMNA_FALLECIDOS).to_pylist()
    total_fall = sum(total_fall)

    # ── Limpiar temporales ─────────────────────────────────────────────────────
    shutil.rmtree(RUTA_TMP, ignore_errors=True)

    if verbose:
        print(f"\n{'=' * 62}")
        print(f"  ✅ Proceso completado")
        print(f"  Filas consolidadas  : {total_filas:,}")
        print(f"  Total de fallecidos : {total_fall:,}")
        print(f"  Archivo guardado en : {ruta_salida}")
        print(f"  Temporales borrados : {RUTA_TMP}")
        print(f"{'=' * 62}\n")


print("Función 'consolidar_defunciones' definida ✓")
print("  → Escribe año a año en disco, combina con PyArrow en streaming.")
print("  → RAM máxima usada: un año + un batch de 500 K filas.")



Función 'consolidar_defunciones' definida ✓
  → Escribe año a año en disco, combina con PyArrow en streaming.
  → RAM máxima usada: un año + un batch de 500 K filas.


## 7. EJECUCIÓN

In [9]:
df_defunciones = consolidar_defunciones(
    ruta_campos  = RUTA_CAMPOS,
    ruta_defun   = RUTA_DEFUN,
    ruta_salida  = RUTA_SALIDA,
    anio_inicio  = ANIO_INICIO,
    anio_fin     = ANIO_FIN,
    verbose      = True,
)


  PROCESAMIENTO DE DEFUNCIONES — COLOMBIA
  Rango        : 1979 – 2024
  Homologación : /kaggle/input/datasets/nicolasacostaa/campos-defunciones/Campos_Defunciones.xlsx
  Defunciones  : /kaggle/input/datasets/nicolasacostaa/defunciones/
  Salida       : /kaggle/working/defunciones_consolidadas.parquet
  Tmp parciales: /kaggle/working/tmp_defun_anios

[1/3] Cargando archivo de homologación...
      16 rangos de formato encontrados.

[2/3] Procesando archivos por año (un año en RAM a la vez)...
  AÑO        ESTADO            FALLECIDOS   FILAS ÚNICAS
  --------   ----------     -------------   ------------
  [OK]      1979:    110,400 fallecidos |    105,373 filas únicas
  [OK]      1980:    125,573 fallecidos |    120,177 filas únicas
  [OK]      1981:    139,505 fallecidos |    132,797 filas únicas
  [OK]      1982:    137,678 fallecidos |    134,922 filas únicas
  [OK]      1983:    140,292 fallecidos |    137,639 filas únicas
  [OK]      1984:    137,189 fallecidos |    132,630 filas

## 8. VERIFICACIÓN DEL RESULTADO

In [10]:
# Vista previa del DataFrame consolidado
print(f"Dimensiones : {df_defunciones.shape[0]:,} filas × {df_defunciones.shape[1]} columnas")
print(f"Columnas    : {list(df_defunciones.columns)}\n")
df_defunciones.head(10)

AttributeError: 'NoneType' object has no attribute 'shape'

In [ ]:
# Resumen de tipos y nulos
df_defunciones.info()


In [ ]:
# Estadísticas de la columna de fallecidos
print(df_defunciones[COLUMNA_FALLECIDOS].describe().to_string())
print(f"\nTotal fallecidos registrados : {df_defunciones[COLUMNA_FALLECIDOS].sum():,}")

In [ ]:
# Verificar el archivo Parquet guardado
df_verificacion = pd.read_parquet(RUTA_SALIDA)
print(f"Parquet leído correctamente: {df_verificacion.shape[0]:,} filas × {df_verificacion.shape[1]} columnas")
df_verificacion.head(5)